In [93]:
import os

os.environ['KERAS_BACKEND'] = 'tensorflow'

import re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import tensorflow as tf
import keras
from keras import layers
from keras.applications import efficientnet
from keras.layers import TextVectorization

from typing import List, Tuple, Dict, Iterable, Optional
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm


keras.utils.set_random_seed(2806)

# Load data

In [76]:
file_path = "../input/curated-cxr-report-generation-dataset/NLP_aug_datasets/df_train_aug.csv"
df = pd.read_csv(file_path, sep = ",")
df

,id,text,path,aug_text
0,s53865364,"In comparison with the study of ___, there is ...",../input/curated-cxr-report-generation-dataset...,['there is no evidence of pneumothorax with th...
1,s56124320,AP chest compared to ___: PICC line ends in th...,../input/curated-cxr-report-generation-dataset...,['AP chest compared to ___: PICC line ends in ...
2,s50991033,The endotracheal tube tip now lies approximate...,../input/curated-cxr-report-generation-dataset...,['endotracheal tube tip now lies approximately...
3,s50337281,"The cardiac, mediastinal and hilar contours ar...",../input/curated-cxr-report-generation-dataset...,"['cardiac, mediastinal and hilar contours are ..."
4,s51904641,Comparison to ___. No relevant change. Low lun...,../input/curated-cxr-report-generation-dataset...,['no relevant change. Low lung volumes. Modera...
...,...,...,...,...
49995,s50858163,1. Interval extubation and removal of the naso...,../input/curated-cxr-report-generation-dataset...,['internal extubation and removal of the nasog...
49996,s52131300,The Swan-Ganz has been removed. Pacemaker defi...,../input/curated-cxr-report-generation-dataset...,['the Swan-Ganz has been removed. the defibril...
49997,s53747282,The Swan-Ganz catheter is been removed. The ri...,../input/curated-cxr-report-generation-dataset...,['the right IJ cordis is in place and the hear...
49998,314,Low lung volumes. Normal heart size. The trach...,../input/curated-cxr-report-generation-dataset...,['the trachea is midline. Lungs are clear. No ...


In [77]:
X = df[['id', 'path', 'aug_text']]
X['aug_text'] = X['aug_text'].str.replace('[\'', '').str.replace('\']', '')
X

/var/folders/my/rlq5shr56m35b7lt22r8gnk40000gn/T/ipykernel_81328/2914708441.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['aug_text'] = X['aug_text'].str.replace('[\'', '').str.replace('\']', '')


,id,path,aug_text
0,s53865364,../input/curated-cxr-report-generation-dataset...,there is no evidence of pneumothorax with the ...
1,s56124320,../input/curated-cxr-report-generation-dataset...,AP chest compared to ___: PICC line ends in th...
2,s50991033,../input/curated-cxr-report-generation-dataset...,endotracheal tube tip now lies approximately 4...
3,s50337281,../input/curated-cxr-report-generation-dataset...,"cardiac, mediastinal and hilar contours are no..."
4,s51904641,../input/curated-cxr-report-generation-dataset...,no relevant change. Low lung volumes. Moderate...
...,...,...,...
49995,s50858163,../input/curated-cxr-report-generation-dataset...,internal extubation and removal of the nasogas...
49996,s52131300,../input/curated-cxr-report-generation-dataset...,the Swan-Ganz has been removed. the defibrilla...
49997,s53747282,../input/curated-cxr-report-generation-dataset...,the right IJ cordis is in place and the heart ...
49998,314,../input/curated-cxr-report-generation-dataset...,the trachea is midline. Lungs are clear. No pn...


# Data preprocessing

In [78]:
def load_captions_data(X):

    """
    Loads captions (text) data and maps them to corresponding image paths.

    Arguments:
        X: dataframe

    Returns:
        caption_mapping: Dictionary mapping image names and the corresponding captions
        text_data: List containing all teh available captions
    """

    caption_mapping = {}
    text_data = []
    images_to_skip = set()

    for i in range (0, len(X) - 1):
        
        # Image name and captions are separated using a tab
        img_name, caption = X['path'][i], X['aug_text'][i]

        # We will remove caption that are either too short or too long
        tokens = caption.strip().split()

        if len(tokens) < 5:
            images_to_skip.add(img_name)
            continue

        if img_name not in images_to_skip:
            # We will add a start and an end token to each caption
            caption = "<start> " + caption.strip() + " <end>"
            text_data.append(caption)

        caption_mapping[img_name] = [caption]

    for img_name in images_to_skip:
        if img_name in caption_mapping:
            del caption_mapping[img_name]

    return caption_mapping, text_data

def train_val_split(caption_data, train_size = 0.9):
    """
    Split the captioning dataset into train and validation sets.

    Arguments:
        caption_data (dict): Dictionary containing the mapped caption data
        train_size (float): Fraction of all the full dataset to use as training data
        shuffle (bool): Whether to shuffle the dataset before splitting

    Returns:
        Training and validation datasets as two separated dicts
    """

    # Get the list of all image paths
    all_images = list(caption_data.keys())

    # Shuffle to ensure randomness
    np.random.shuffle(all_images)

    # Split into training and validation sets
    train_size = int(len(caption_data) * train_size)

    training_data = {
        img_name: caption_data[img_name] for img_name in all_images[:train_size]
    }

    validation_data = {
        img_name: caption_data[img_name] for img_name in all_images[train_size:]
    }

    # Return two sets
    return training_data, validation_data


In [79]:
captions_mapping, text_data = load_captions_data(X)

# Split the dataset into training and validation sets
train_data, val_test_data = train_val_split(captions_mapping)

print("Number of training samples: ", len(train_data))

val_data, test_data = train_val_split(val_test_data, train_size = 0.5)

print("Number of validation samples: ", len(val_data))
print("Number of test samples: ", len(test_data))

Number of training samples:  44993
Number of validation samples:  2500
Number of test samples:  2500


# Model

In [ ]:
# Vocabulary size
VOCAB_SIZE = 20000

# Fixed length allowed for any sequence
SEQ_LENGTH = 50

# Dimension for the image embeddings and token embeddings
EMBED_DIM = 512

# Per-layer units in the feed forward network
FF_DIM = 512

# Other training parameters
BATCH_SIZE = 64
EPOCHS = 50
AUTOTUNE = tf.data.AUTOTUNE

In [117]:
# Load the model
if (torch.cuda.is_available()):
    device = torch.device("cuda")
elif (torch.backends.mps.is_available()):
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model, preprocess = clip.load('ViT-B/32', device)
print(f"Using device: {device}")

Using device: mps


In [122]:
class CXRDictDataset(Dataset):
    def __init__(self, data_dict, preprocess, tokenizer=clip.tokenize):
        self.items = list(data_dict.items())  # [(img_path, caption), ...]
        self.preprocess = preprocess
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, text = self.items[idx]
        image = self.preprocess(Image.open(path).convert("RGB"))
        text_token = self.tokenizer(text, truncate=True)[0]
        return image, text_token


In [123]:
def collate_fn(batch):
    images, texts = zip(*batch)
    images = torch.stack(images, dim=0)
    texts  = torch.stack(texts, dim=0)   # stack the token tensors
    return images, texts

In [124]:
# Transfer our train/test/val dataset into our new class
train_ds = CXRDictDataset(train_data, preprocess)
val_ds   = CXRDictDataset(val_data, preprocess)
test_ds = CXRDictDataset(test_data, preprocess)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn = collate_fn)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn = collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn = collate_fn)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.02)


In [125]:
images, texts = next(iter(train_loader))
print(images.shape, texts.shape)


torch.Size([64, 3, 224, 224]) torch.Size([64, 77])


In [126]:
def train_clip_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for images, texts in tqdm(loader):
        images, texts = images.to(device), texts.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type=="mps")):
            img_feats = model.encode_image(images)
            txt_feats = model.encode_text(texts)

            img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
            txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)

            logit_scale = model.logit_scale.exp()
            logits_i2t = logit_scale * img_feats @ txt_feats.t()
            logits_t2i = logits_i2t.t()

            targets = torch.arange(images.size(0), device=device)
            loss = (F.cross_entropy(logits_i2t, targets) + F.cross_entropy(logits_t2i, targets)) / 2

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


In [127]:
for epoch in range(2):
    loss = train_clip_epoch(model, train_loader, optimizer, device)
    print(f"Epoch {epoch+1}: loss={loss:.4f}")

100%|██████████| 704/704 [18:43<00:00,  1.60s/it]


Epoch 1: loss=nan


100%|██████████| 704/704 [19:45<00:00,  1.68s/it]

Epoch 2: loss=nan


In [129]:
@torch.no_grad()
def build_embeddings(model, preprocess, data_dict, device):
    image_embs, text_embs, paths, texts = [], [], [], []
    for path, text in tqdm(data_dict.items()):
        image = preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
        text_tok = clip.tokenize([text], truncate=True).to(device)

        img_feat = model.encode_image(image)
        txt_feat = model.encode_text(text_tok)
        img_feat /= img_feat.norm(dim=-1, keepdim=True)
        txt_feat /= txt_feat.norm(dim=-1, keepdim=True)

        image_embs.append(img_feat)
        text_embs.append(txt_feat)
        paths.append(path)
        texts.append(text)

    return torch.cat(image_embs), torch.cat(text_embs), paths, texts

image_embs, text_embs, paths, texts = build_embeddings(model, preprocess, val_data, device)


  0%|          | 0/2500 [00:00<?, ?it/s]


AttributeError: 'list' object has no attribute 'find'